In [94]:
import xml.etree.ElementTree as ET
import copy

In [95]:
persontrips_filepath = "D:\\Dokumente\\sumo\\maloja\\12_non_hotel\\persontrips_maloja_non_hotel_winter.xml"
persontrips_filepath_output = "D:\\Dokumente\\sumo\\maloja\\12_non_hotel\\persontrips_maloja_non_hotel_winter_modded.xml"
routes_filepath = "D:\\Dokumente\\sumo\\maloja\\12_non_hotel\\persontrips_maloja_non_hotel_resolved_winter_bus.rou.xml"
routes_filepath_output = "D:\\Dokumente\\sumo\\maloja\\12_non_hotel\\persontrips_maloja_non_hotel_resolved_winter_bus_modded.rou.xml"

In [96]:
demand_arriving_leaving = {
    "taz_St_Moritz_Sink-taz_Bregaglia": 22,
    "taz_St_Moritz_Sink-taz_Silvaplana": 36,
    "taz_St_Moritz_Sink-taz_Sils_Engadin": 75,
    "taz_St_Moritz_Sink-taz_St_Moritz": 290
}

demand_shopping = {
    "taz_Bregaglia-taz_shopping_bregaglia_vicosoprano": 4,
    "taz_Silvaplana-taz_shopping_silvaplana": 7,
    "taz_Sils_Engadin-taz_shopping_sils_engadin": 15,
    "taz_St_Moritz-taz_shopping_st_moritz_west": 58
}

demand_activities = {
    "taz_Bregaglia-taz_ski_aela": 76,
    
    "taz_Bregaglia-taz_St_Moritz": 6,
    "taz_Bregaglia-taz_Bregaglia": 6,
    "taz_Bregaglia-taz_Silvaplana": 6,
    "taz_Bregaglia-taz_Sils_Engadin": 6,
    "taz_Bregaglia-taz_St_Moritz_Sink": 6,

    "taz_Silvaplana-taz_ski_corvatsch_silvaplana": 154,

    "taz_Silvaplana-taz_St_Moritz": 13,
    "taz_Silvaplana-taz_Bregaglia": 13,
    "taz_Silvaplana-taz_Silvaplana": 13,
    "taz_Silvaplana-taz_Sils_Engadin": 13,
    "taz_Silvaplana-taz_St_Moritz_Sink": 13,
    
    "taz_Sils_Engadin-taz_ski_corvatsch_sils_im_engadin": 389,

    "taz_Sils_Engadin-taz_St_Moritz": 33,
    "taz_Sils_Engadin-taz_Bregaglia": 33,
    "taz_Sils_Engadin-taz_Silvaplana": 33,
    "taz_Sils_Engadin-taz_Sils_Engadin": 33,
    "taz_Sils_Engadin-taz_St_Moritz_Sink": 33,
    
    "taz_St_Moritz-taz_ski_corviglia_piz_nair_st_moritz": 1294,

    "taz_St_Moritz-taz_St_Moritz": 111,
    "taz_St_Moritz-taz_Bregaglia": 111,
    "taz_St_Moritz-taz_Silvaplana": 111,
    "taz_St_Moritz-taz_Sils_Engadin": 111,
    "taz_St_Moritz-taz_St_Moritz_Sink": 111,
}

In [97]:
demand_non_hotel_arriving = copy.deepcopy(demand_arriving_leaving)
demand_non_hotel_leaving = copy.deepcopy(demand_arriving_leaving)

In [98]:
demand_shopping_from_non_hotel = copy.deepcopy(demand_shopping)
demand_shopping_to_non_hotel = copy.deepcopy(demand_shopping)

In [99]:
demand_from_non_hotel = copy.deepcopy(demand_activities)
demand_to_non_hotel = copy.deepcopy(demand_activities)

In [100]:
persontrips_file_xmltree = ET.parse(persontrips_filepath)
persontrips_file_xmlroot = persontrips_file_xmltree.getroot()

In [101]:
routes_file_xmltree = ET.parse(routes_filepath)
routes_file_xmlroot = routes_file_xmltree.getroot()

In [102]:
for trip_child in list(routes_file_xmlroot):
    has_ride = False
    for stage_child in list(trip_child):
        if stage_child.tag == "ride":
            if "intended" in stage_child.attrib:
                has_ride = True
                break

    if has_ride == True:
        for persontrip_child in list(persontrips_file_xmlroot):
            if trip_child.get("id") == persontrip_child.get("id"):
                demand_ft_key = str(persontrip_child[0].get("fromTaz")) + "-" + str(persontrip_child[0].get("toTaz"))
                demand_tf_key = str(persontrip_child[0].get("toTaz")) + "-" + str(persontrip_child[0].get("fromTaz"))

                if 36000 <= float(str(trip_child.get("depart"))) <= 43200 and demand_tf_key in demand_non_hotel_leaving and demand_non_hotel_leaving[demand_tf_key] > 0:
                    demand_non_hotel_leaving[demand_tf_key] = demand_non_hotel_leaving[demand_tf_key] - 1
                    break
                elif 43200 <= float(str(trip_child.get("depart"))) <= 50400 and demand_ft_key in demand_non_hotel_arriving and demand_non_hotel_arriving[demand_ft_key] > 0:
                    demand_non_hotel_arriving[demand_ft_key] = demand_non_hotel_arriving[demand_ft_key] - 1
                    break
                elif 50400 <= float(str(trip_child.get("depart"))) <= 64800 and demand_ft_key in demand_shopping_from_non_hotel and demand_shopping_from_non_hotel[demand_ft_key] > 0:
                    demand_shopping_from_non_hotel[demand_ft_key] = demand_shopping_from_non_hotel[demand_ft_key] - 1
                    break
                elif 54000 <= float(str(trip_child.get("depart"))) <= 66600 and demand_tf_key in demand_shopping_to_non_hotel and demand_shopping_to_non_hotel[demand_tf_key] > 0:
                    demand_shopping_to_non_hotel[demand_tf_key] = demand_shopping_to_non_hotel[demand_tf_key] - 1
                    break
                elif demand_ft_key in demand_from_non_hotel and demand_from_non_hotel[demand_ft_key] > 0:
                    demand_from_non_hotel[demand_ft_key] = demand_from_non_hotel[demand_ft_key] - 1
                    break
                elif demand_tf_key in demand_to_non_hotel and demand_to_non_hotel[demand_tf_key] > 0:
                    demand_to_non_hotel[demand_tf_key] = demand_to_non_hotel[demand_tf_key] - 1
                    break
                else:
                    routes_file_xmlroot.remove(trip_child)
                    break

    if has_ride == False:
        routes_file_xmlroot.remove(trip_child)

In [103]:
demand_non_hotel_leaving

{'taz_St_Moritz_Sink-taz_Bregaglia': 0,
 'taz_St_Moritz_Sink-taz_Silvaplana': 0,
 'taz_St_Moritz_Sink-taz_Sils_Engadin': 0,
 'taz_St_Moritz_Sink-taz_St_Moritz': 0}

In [104]:
demand_non_hotel_arriving

{'taz_St_Moritz_Sink-taz_Bregaglia': 0,
 'taz_St_Moritz_Sink-taz_Silvaplana': 0,
 'taz_St_Moritz_Sink-taz_Sils_Engadin': 0,
 'taz_St_Moritz_Sink-taz_St_Moritz': 0}

In [105]:
demand_shopping_from_non_hotel

{'taz_Bregaglia-taz_shopping_bregaglia_vicosoprano': 0,
 'taz_Silvaplana-taz_shopping_silvaplana': 0,
 'taz_Sils_Engadin-taz_shopping_sils_engadin': 0,
 'taz_St_Moritz-taz_shopping_st_moritz_west': 0}

In [106]:
demand_shopping_to_non_hotel

{'taz_Bregaglia-taz_shopping_bregaglia_vicosoprano': 0,
 'taz_Silvaplana-taz_shopping_silvaplana': 0,
 'taz_Sils_Engadin-taz_shopping_sils_engadin': 0,
 'taz_St_Moritz-taz_shopping_st_moritz_west': 0}

In [107]:
demand_from_non_hotel

{'taz_Bregaglia-taz_ski_aela': 0,
 'taz_Bregaglia-taz_St_Moritz': 0,
 'taz_Bregaglia-taz_Bregaglia': 0,
 'taz_Bregaglia-taz_Silvaplana': 0,
 'taz_Bregaglia-taz_Sils_Engadin': 0,
 'taz_Bregaglia-taz_St_Moritz_Sink': 0,
 'taz_Silvaplana-taz_ski_corvatsch_silvaplana': 0,
 'taz_Silvaplana-taz_St_Moritz': 0,
 'taz_Silvaplana-taz_Bregaglia': 0,
 'taz_Silvaplana-taz_Silvaplana': 0,
 'taz_Silvaplana-taz_Sils_Engadin': 0,
 'taz_Silvaplana-taz_St_Moritz_Sink': 0,
 'taz_Sils_Engadin-taz_ski_corvatsch_sils_im_engadin': 0,
 'taz_Sils_Engadin-taz_St_Moritz': 0,
 'taz_Sils_Engadin-taz_Bregaglia': 0,
 'taz_Sils_Engadin-taz_Silvaplana': 0,
 'taz_Sils_Engadin-taz_Sils_Engadin': 0,
 'taz_Sils_Engadin-taz_St_Moritz_Sink': 0,
 'taz_St_Moritz-taz_ski_corviglia_piz_nair_st_moritz': 0,
 'taz_St_Moritz-taz_St_Moritz': 0,
 'taz_St_Moritz-taz_Bregaglia': 0,
 'taz_St_Moritz-taz_Silvaplana': 0,
 'taz_St_Moritz-taz_Sils_Engadin': 0,
 'taz_St_Moritz-taz_St_Moritz_Sink': 0}

In [108]:
demand_to_non_hotel

{'taz_Bregaglia-taz_ski_aela': 0,
 'taz_Bregaglia-taz_St_Moritz': 0,
 'taz_Bregaglia-taz_Bregaglia': 0,
 'taz_Bregaglia-taz_Silvaplana': 0,
 'taz_Bregaglia-taz_Sils_Engadin': 0,
 'taz_Bregaglia-taz_St_Moritz_Sink': 0,
 'taz_Silvaplana-taz_ski_corvatsch_silvaplana': 0,
 'taz_Silvaplana-taz_St_Moritz': 0,
 'taz_Silvaplana-taz_Bregaglia': 0,
 'taz_Silvaplana-taz_Silvaplana': 0,
 'taz_Silvaplana-taz_Sils_Engadin': 0,
 'taz_Silvaplana-taz_St_Moritz_Sink': 0,
 'taz_Sils_Engadin-taz_ski_corvatsch_sils_im_engadin': 0,
 'taz_Sils_Engadin-taz_St_Moritz': 0,
 'taz_Sils_Engadin-taz_Bregaglia': 0,
 'taz_Sils_Engadin-taz_Silvaplana': 0,
 'taz_Sils_Engadin-taz_Sils_Engadin': 0,
 'taz_Sils_Engadin-taz_St_Moritz_Sink': 0,
 'taz_St_Moritz-taz_ski_corviglia_piz_nair_st_moritz': 0,
 'taz_St_Moritz-taz_St_Moritz': 0,
 'taz_St_Moritz-taz_Bregaglia': 0,
 'taz_St_Moritz-taz_Silvaplana': 0,
 'taz_St_Moritz-taz_Sils_Engadin': 0,
 'taz_St_Moritz-taz_St_Moritz_Sink': 0}

In [109]:
for persontrip_child in list(persontrips_file_xmlroot):
    persontrip_in_routes = False
    for trip_child in list(routes_file_xmlroot):
        if trip_child.get("id") == persontrip_child.get("id"):
            persontrip_in_routes = True
            break
    
    if persontrip_in_routes == False:
        persontrips_file_xmlroot.remove(persontrip_child)

In [110]:
persontrips_file_xmltree.write(persontrips_filepath_output)
routes_file_xmltree.write(routes_filepath_output)